In [16]:
import pandas as pd
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import explode, expr, collect_list
from pyspark.mllib.evaluation import RankingMetrics

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [3]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [4]:
meta_schema = StructType([
    StructField("main_category", StringType(), True),
    StructField("title", StringType(), True),
    StructField("average_rating", DoubleType(), True),
    StructField("rating_number", LongType(), True),
    StructField("features", ArrayType(StringType()), True),
    StructField("description", ArrayType(StringType()), True),
    StructField("price", DoubleType(), True),
    StructField("images", ArrayType(MapType(StringType(), StringType())), True),
    StructField("videos", ArrayType(MapType(StringType(), StringType())), True),
    StructField("store", StringType(), True),
    StructField("categories", ArrayType(StringType()), True),

    StructField("details", StringType(), True),

    StructField("parent_asin", StringType(), True),
    StructField("bought_together", DoubleType(), True),
    StructField("subtitle", StringType(), True),
    StructField("author", StringType(), True),
])
raw_meta_df = spark.read.text("/content/drive/MyDrive/data - MoMD/meta_Office_Products.jsonl")
meta_df = (
    raw_meta_df
    .select(from_json(col("value"), meta_schema).alias("data"))
    .select("data.*")
)

In [5]:
df_data = spark.read.format("json").option("inferSchema", "true").option("header", "true").load("/content/drive/MyDrive/data - MoMD/Office_Products.jsonl")

In [ ]:
meta_df.show()

+--------------------+--------------------+--------------+-------------+---------------------+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+---------------+--------------------+------+
|       main_category|               title|average_rating|rating_number|             features|         description| price|              images|              videos|               store|          categories|             details|parent_asin|bought_together|            subtitle|author|
+--------------------+--------------------+--------------+-------------+---------------------+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+---------------+--------------------+------+
|     Office Products|Alliance Rubber 0...|           4.5|          665| [REUSABLE: These ...|[Alliance Rubber ...|  2.68|[{thumb -> https:...|[{tit

In [ ]:
df_data.show()

+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|      asin|helpful_vote|              images|parent_asin|rating|                text|    timestamp|               title|             user_id|verified_purchase|
+----------+------------+--------------------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+
|B01AHHL4X2|           0|                  []| B01MZ3SD2X|   5.0|Lovely ink. Write...|1677939345945| Pretty & I love it!|AFKZENTNBQ7A7V7UX...|             true|
|B08L6H23JZ|           0|                  []| B08L6H23JZ|   4.0|Overall I’m prett...|1677939160682|2 excellent 1 ext...|AFKZENTNBQ7A7V7UX...|             true|
|B07JDZ5J46|           2|                  []| B07JDZ5J46|   1.0|[[VIDEOID:63276c1...|1660188831933|I don’t get the r...|AFKZENTNBQ7A7V7UX...|             true|
|B004MNX7EW|           0|[{IMAGE, 

Lọc cột cần thiết

In [7]:
meta_df_raw = meta_df
df_data_raw = df_data

In [8]:
df_data = df_data.select("asin","parent_asin","rating","timestamp","user_id","verified_purchase")

In [11]:
df_data.printSchema()

root
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [10]:
df_data.count()

12845712

In [11]:
df_data = df_data.withColumnsRenamed({
    'asin':'product_id',
    'parent_asin':'product_parent',
    'rating':'review_rating',
    'timestamp':'review_date'
})

In [12]:
user_counts = df_data.groupBy("user_id").agg(count("*").alias("number_rating"))
active_users = user_counts.filter(col("number_rating") >= 5)
active_users = active_users.drop("number_rating")
df_data = df_data.join(active_users, "user_id")

In [13]:
product_counts = df_data.groupBy("product_id").agg(count("*").alias("number_rating"))
popular_products = product_counts.filter(col("number_rating") >= 5)
popular_products = popular_products.drop("number_rating")
df_data = df_data.join(popular_products, "product_id")

In [15]:
df_data.count()

2165547

In [17]:
output_path_rating = "/content/drive/MyDrive/data_parquet/RS/rating_rs.parquet"
df_data.write.mode("overwrite").parquet(output_path_rating)

In [9]:
meta_df.count()

710503

In [18]:
meta_df.printSchema()

root
 |-- main_category: string (nullable = true)
 |-- title: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- price: double (nullable = true)
 |-- parent_asin: string (nullable = true)



In [32]:
meta_df = meta_df.select(
    col("parent_asin").alias("product_id"),
    "title",
    "description",
    "features",
    "categories",
    "store"
)

In [21]:
meta_df.show(truncate=False)

+----------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [24]:
total_meta = meta_df.count()

In [33]:
null_meta_counts = meta_df.select([
    sum(
        col(c).isNull().cast('int')
    ).alias(c)
    for c in meta_df.columns
])

null_meta_counts.select([
    concat(
        col(c).cast('string'),
        lit(" ("),
        round(col(c).cast('float') * 100 / total_meta, 2).cast('string'),
        lit("%)")
    ).alias(c)
    for c in meta_df.columns
]).show()

+----------+--------+-----------+--------+----------+-------------+
|product_id|   title|description|features|categories|        store|
+----------+--------+-----------+--------+----------+-------------+
|  0 (0.0%)|0 (0.0%)|   0 (0.0%)|0 (0.0%)|  0 (0.0%)|12251 (1.72%)|
+----------+--------+-----------+--------+----------+-------------+



In [34]:
meta_df = meta_df.fillna({
    "store": "unknown"
})

In [35]:
output_path_meta = "/content/drive/MyDrive/data_parquet/RS/meta_rs.parquet"
df_data.write.mode("overwrite").parquet(output_path_meta)